# Lesson 06 — Empirical Mode Decomposition (EMD)
**TTK4260: Multivariate Data Analysis and Machine Learning**  
*Department of Engineering Cybernetics, NTNU*

---

## Learning objectives

By the end of this notebook you will be able to:

1. Explain **why** EMD is preferred over FFT and wavelets for non-stationary signals.
2. State the two conditions that define an **Intrinsic Mode Function (IMF)**.
3. Trace through the **sifting algorithm** step by step.
4. Implement a clean, well-documented Python version of EMD from scratch.
5. Apply EMD to a real signal and interpret the resulting components.
6. Understand how the **stopping criterion** controls decomposition quality.

---

## Outline

| Section | Topic |
|---------|-------|
| 1 | Background and motivation |
| 2 | IMF conditions |
| 3 | The sifting algorithm |
| 4 | Python implementation |
| 5 | Example decomposition |
| 6 | Step-by-step visualisation |
| 7 | Sensitivity to stopping criterion |
| 8 | Reconstruction check |
| 9 | Summary |


---
## 1. Background and motivation

### Why not FFT?

The Fourier Transform represents a signal as a sum of sinusoids of **fixed frequency**. This works well when the signal is *stationary* — i.e. its frequency content does not change over time.

Most real-world signals violate this assumption:
- Neural LFP recordings change their dominant rhythm with cognitive state
- Gearbox vibrations shift frequency as load changes
- Ocean waves are driven by multiple intermittent sources

Applying FFT to a non-stationary signal smears time-localised events across the entire spectrum, making interpretation unreliable.

### Why not wavelets?

Wavelets improve on FFT by providing time–frequency localisation, but still require choosing a **fixed mother wavelet** $\psi$ before analysis. A poor choice introduces artefacts, and the Heisenberg–Gabor uncertainty $\Delta t \cdot \Delta f \geq \tfrac{1}{4\pi}$ places a hard limit on simultaneous time and frequency resolution.

### EMD: the data-driven alternative

> **Empirical Mode Decomposition** (Huang et al., 1998) adaptively decomposes a signal into oscillatory components called **Intrinsic Mode Functions (IMFs)** directly in the time domain — no basis functions, no stationarity assumption.

$$x(t) = \sum_{i=1}^{n} \text{IMF}_i(t) + r(t)$$

where $r(t)$ is the residual trend.

| Property | FFT | Wavelet | EMD |
|----------|-----|---------|-----|
| Basis | Fixed sine | Fixed $\psi$ | Adaptive |
| Time resolution | None | Multi-scale | Full |
| Stationarity required | Yes | No | No |
| Handles nonlinear dynamics | No | No | Yes |

**Advantages of EMD**
- Non-stationary frequency characteristics are preserved
- Instantaneous frequency via the Hilbert transform yields physically meaningful results
- Operates entirely in the time domain

**Limitations to keep in mind**
- The stopping criterion `stoplim` is a free parameter
- Mode mixing: oscillations of interest may spread across multiple IMFs
- Boundary effects at both ends of the signal


---
## 2. What is an Intrinsic Mode Function?

A function $c(t)$ qualifies as an IMF if and only if it satisfies **both** conditions:

> **Condition 1 — Extrema / zero-crossing balance:**  
> The number of local extrema and zero-crossings must differ by at most one across the entire signal.

> **Condition 2 — Local symmetry (the key condition in practice):**  
> The mean of the upper envelope $u(t)$ (through local maxima) and the lower envelope $\ell(t)$ (through local minima) is approximately zero everywhere:
> $$\frac{u(t) + \ell(t)}{2} \approx 0 \quad \forall\, t$$

These conditions ensure each IMF is a **narrow-band, near-monocomponent** oscillation, making the Hilbert instantaneous frequency physically interpretable:

$$\omega_i(t) = \frac{d}{dt} \arg\!\left[\mathcal{H}\{\text{IMF}_i(t)\}\right]$$


---
## 3. The sifting algorithm

To extract one IMF from signal $x(t)$, repeat the following steps until Condition 2 is satisfied:

```
h_0 = x(t)

repeat:
    1. Find all local maxima (peaks) and minima (troughs) of h_k

    2. Cubic-spline the peaks   → upper envelope u(t)
       Cubic-spline the troughs → lower envelope l(t)

    3. Compute the mean envelope:
           m(t) = [u(t) + l(t)] / 2

    4. Subtract the mean:
           h_{k+1} = h_k − m(t)

    5. Evaluate the stopping criterion:
           sdk = sum|m(t)|^2 / sum|h_k(t)|^2   (over interior samples)

    if sdk < stoplim:  accept h_k as IMF_i  → break

IMF_i    = h_k
residual = x − IMF_i
→ repeat on residual to find IMF_{i+1}
```

The **stopping criterion** `stoplim` controls how flat the mean envelope must be before we accept a candidate as an IMF. Smaller values → stricter IMF, more iterations, but also higher risk of over-sifting.


---
## 4. Implementation

### 4.1 Imports


In [ ]:
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

import numpy as np
import scipy as sp
from scipy import signal, interpolate
import matplotlib.pyplot as plt

# Consistent, clean plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Libraries loaded successfully.')


### 4.2 Helper functions

Two small helpers keep the main sifting loop readable:

- `_emd_complim` — clamps the mean envelope at boundary samples to avoid   wild cubic-spline extrapolation outside the range of observed extrema.
- `_emd_comperror` — computes the normalised stopping criterion $sdk$,   evaluated only over the interior region between the innermost extrema.


In [ ]:
def _emd_complim(mean_t, pks, trs):
    """Clamp mean envelope to boundary values outside the range of extrema.

    Parameters
    ----------
    mean_t : ndarray  Current mean envelope (same length as signal).
    pks    : ndarray  Sample indices of local maxima.
    trs    : ndarray  Sample indices of local minima.

    Returns
    -------
    mean_t : ndarray  Mean envelope with boundaries clamped.
    """
    samp_start = int(np.max((np.min(pks), np.min(trs))))
    samp_end   = int(np.min((np.max(pks), np.max(trs)))) + 1
    mean_t[:samp_start] = mean_t[samp_start]
    mean_t[samp_end:]   = mean_t[samp_end]
    return mean_t


def _emd_comperror(h, mean, pks, trs):
    """Compute the normalised sifting error (stopping criterion sdk).

    Only the interior region between the innermost peak/trough pair is used,
    avoiding boundary contamination.

    Parameters
    ----------
    h    : ndarray  Current candidate signal.
    mean : ndarray  Mean of upper and lower envelopes.
    pks  : ndarray  Peak indices.
    trs  : ndarray  Trough indices.

    Returns
    -------
    sdk : float  Normalised error.  Accept as IMF when sdk < stoplim.
    """
    samp_start = int(np.max((np.min(pks), np.min(trs))))
    samp_end   = int(np.min((np.max(pks), np.max(trs)))) + 1
    return (np.sum(mean[samp_start:samp_end] ** 2)
            / np.sum(h[samp_start:samp_end] ** 2))


### 4.3 Main EMD function

Each numbered comment maps directly to one step of the pseudocode in Section 3.


In [ ]:
def emd(x, n_imf=3, stoplim=1e-3):
    """Decompose signal x into n_imf IMFs via Empirical Mode Decomposition.

    Parameters
    ----------
    x       : array_like  Input signal (1-D, real-valued).
    n_imf   : int         Number of IMFs to extract.  Default: 3.
    stoplim : float       Stopping criterion threshold.  Default: 1e-3.

    Returns
    -------
    imfs     : ndarray, shape (n_imf,)  IMFs, highest frequency first.
    residual : ndarray                  Residual trend.
    """
    x        = np.asarray(x, dtype=float)
    t        = np.arange(len(x))
    imfs     = np.empty(n_imf, dtype=object)
    residual = x.copy()

    for i in range(n_imf):
        candidate = residual.copy()

        while True:
            # Step 1: find local extrema
            pks = sp.signal.argrelmax(candidate)[0]
            trs = sp.signal.argrelmin(candidate)[0]
            if len(pks) < 2 or len(trs) < 2:
                break  # not enough extrema

            # Step 2: cubic-spline envelopes
            upper_env = sp.interpolate.InterpolatedUnivariateSpline(
                pks, candidate[pks], k=3)(t)
            lower_env = sp.interpolate.InterpolatedUnivariateSpline(
                trs, candidate[trs], k=3)(t)

            # Step 3: mean envelope
            mean_env = (upper_env + lower_env) / 2
            mean_env = _emd_complim(mean_env, pks, trs)

            # Step 4: subtract mean
            candidate_new = candidate - mean_env

            # Step 5: evaluate stopping criterion
            sdk = _emd_comperror(candidate, mean_env, pks, trs)
            candidate = candidate_new
            if sdk < stoplim:
                break

        imfs[i]  = candidate
        residual = residual - candidate

    return imfs, residual


---
## 5. Example decomposition

### 5.1 Load signal

The example signal is a 2-second segment of a local field potential (LFP) recording sampled at 1 kHz. It contains multiple oscillatory rhythms superimposed on a slow drift — a typical non-stationary biomedical signal.


In [ ]:
x_full = np.load('./exampledata.npy')
x = x_full[7000:9001].astype(float)
t = np.arange(len(x)) * 1e-3   # time axis in seconds (1 kHz)

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(t, x, color='steelblue', lw=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage (µV)')
ax.set_title('Raw LFP signal — 2-second window')
plt.tight_layout()
plt.show()

print(f'Length : {len(x)} samples ({t[-1]:.3f} s)')
print(f'Range  : [{x.min():.1f}, {x.max():.1f}] µV')


### 5.2 Run EMD and plot all IMFs

The grey trace in each panel is the original signal shown for reference, so you can appreciate which aspect of the signal each IMF captures.


In [ ]:
N_IMF = 5
imfs, residual = emd(x, n_imf=N_IMF, stoplim=1e-3)

fig, axes = plt.subplots(N_IMF + 2, 1, figsize=(12, 14), sharex=True)
colors = plt.cm.tab10.colors

axes[0].plot(t, x, color='steelblue', lw=0.8)
axes[0].set_ylabel('Original')
axes[0].set_title('EMD decomposition — 5 IMFs')

for i, imf in enumerate(imfs):
    axes[i + 1].plot(t, x,   color='0.80', lw=0.6, zorder=0)
    axes[i + 1].plot(t, imf, color=colors[i], lw=0.9, zorder=1)
    axes[i + 1].set_ylabel(f'IMF {i+1}')
    axes[i + 1].set_ylim(-1000, 1000)

axes[-1].plot(t, residual, color='dimgray', lw=0.9)
axes[-1].set_ylabel('Residual')
axes[-1].set_ylim(-1000, 1000)
axes[-1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()


**Observations to note:**

- **IMF 1** captures the highest-frequency oscillations.
- Each subsequent IMF contains progressively lower-frequency content.
- The **residual** captures the slow trend (drift) that is not oscillatory.

> 💡 Unlike FFT, the frequency of each IMF is **not fixed** — it varies in time. This is the key advantage of EMD for non-stationary signals.


---
## 6. Step-by-step sifting visualisation

This section walks through each sub-operation of the sifting algorithm so you can build a precise mental model.


### Step 1 — Find local extrema

`scipy.signal.argrelmax` and `argrelmin` return the indices of peaks and troughs.


In [ ]:
x_temp = x.copy()
pks = sp.signal.argrelmax(x_temp)[0]
trs = sp.signal.argrelmin(x_temp)[0]

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(t, x_temp, 'k', lw=0.7, label='signal')
ax.plot(t[pks], x_temp[pks], 'r.', ms=4, label=f'maxima ({len(pks)})')
ax.plot(t[trs], x_temp[trs], 'b.', ms=4, label=f'minima ({len(trs)})')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Voltage (µV)')
ax.set_title('Step 1: local extrema')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

print(f'Found {len(pks)} peaks and {len(trs)} troughs.')


### Step 2 — Cubic-spline envelopes

Connecting peaks with a cubic spline → upper envelope $u(t)$; connecting troughs → lower envelope $\ell(t)$.


In [ ]:
t_idx = np.arange(len(x))
upper_env = sp.interpolate.InterpolatedUnivariateSpline(
    pks, x_temp[pks], k=3)(t_idx)
lower_env = sp.interpolate.InterpolatedUnivariateSpline(
    trs, x_temp[trs], k=3)(t_idx)

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(t, x_temp, 'k', lw=0.7, label='signal', zorder=3)
ax.plot(t[pks], x_temp[pks], 'r.', ms=4, zorder=4)
ax.plot(t[trs], x_temp[trs], 'b.', ms=4, zorder=4)
ax.plot(t, upper_env, 'r-', lw=1.4, label='upper envelope $u(t)$')
ax.plot(t, lower_env, 'b-', lw=1.4, label='lower envelope $\\ell(t)$')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Voltage (µV)')
ax.set_title('Step 2: cubic-spline envelopes')
ax.set_ylim(-1000, 1000); ax.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()


### Step 3 — Subtract the mean envelope

$m(t) = \tfrac{u(t) + \ell(t)}{2}$.  The candidate $h(t) = x(t) - m(t)$ should have $m \approx 0$ when it is a true IMF.


In [ ]:
mean_env  = (upper_env + lower_env) / 2
mean_env  = _emd_complim(mean_env, pks, trs)
candidate = x_temp - mean_env

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)

axes[0].plot(t, x_temp, 'k', lw=0.7, label='signal')
axes[0].plot(t, upper_env, 'r', lw=1.2, label='$u(t)$')
axes[0].plot(t, lower_env, 'b', lw=1.2, label='$\\ell(t)$')
axes[0].plot(t, mean_env, color='darkorange', lw=1.8, label='mean $m(t)$')
axes[0].set_ylabel('Voltage (µV)'); axes[0].set_ylim(-1000, 1000)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_title('Step 3a: envelopes and mean')

axes[1].plot(t, x_temp,   color='0.75', lw=0.7, label='original')
axes[1].plot(t, candidate, 'k',         lw=0.9, label='candidate $h = x - m$')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Voltage (µV)')
axes[1].set_ylim(-1000, 1000); axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_title('Step 3b: candidate after mean subtraction')

plt.tight_layout(); plt.show()


### Step 4 — Check the stopping criterion

$sdk = \frac{\sum |m(t)|^2}{\sum |h(t)|^2}$ (over the interior region).

- $sdk < \texttt{stoplim}$ → accept as IMF
- Otherwise → set $h \leftarrow h - m$ and repeat from Step 1


In [ ]:
sdk = _emd_comperror(x_temp, mean_env, pks, trs)

print(f'sdk     = {sdk:.6f}')
print(f'stoplim = {1e-3:.6f}')
print()
if sdk < 1e-3:
    print('sdk < stoplim  ->  candidate IS an IMF on the first iteration.')
else:
    print('sdk >= stoplim  ->  continue sifting (subtract mean and repeat).')


### 6.5 Watching the sifting iterations converge

The function below runs the full sifting loop for one IMF and plots each intermediate candidate so you can watch the mean envelope progressively flatten toward zero.


In [ ]:
def sifting_iterations(x, stoplim=1e-3, max_iter=80, figwidth=13):
    """Run the sifting process for one IMF and plot every iteration.

    Parameters
    ----------
    x        : ndarray  Input signal.
    stoplim  : float    Stopping threshold.
    max_iter : int      Safety cap on iterations.
    figwidth : float    Width of each iteration panel.

    Returns
    -------
    imf    : ndarray  Extracted IMF.
    n_iter : int      Number of sifting iterations performed.
    """
    t_idx     = np.arange(len(x))
    t_s       = t_idx * 1e-3
    candidate = x.copy()
    history   = []

    for k in range(max_iter):
        pks = sp.signal.argrelmax(candidate)[0]
        trs = sp.signal.argrelmin(candidate)[0]
        if len(pks) < 2 or len(trs) < 2:
            break

        upper_env = sp.interpolate.InterpolatedUnivariateSpline(
            pks, candidate[pks], k=3)(t_idx)
        lower_env = sp.interpolate.InterpolatedUnivariateSpline(
            trs, candidate[trs], k=3)(t_idx)

        mean_env  = (upper_env + lower_env) / 2
        mean_env  = _emd_complim(mean_env, pks, trs)
        sdk       = _emd_comperror(candidate, mean_env, pks, trs)

        history.append({'k': k+1, 'candidate': candidate.copy(),
                        'mean': mean_env.copy(), 'sdk': sdk})
        candidate = candidate - mean_env
        if sdk < stoplim:
            break

    n_iter = len(history)
    fig, axes = plt.subplots(n_iter, 1,
                             figsize=(figwidth, 2.2 * n_iter),
                             sharex=True)
    if n_iter == 1:
        axes = [axes]

    for ax, h in zip(axes, history):
        ax.plot(t_s, x,              color='0.75', lw=0.6, label='original')
        ax.plot(t_s, h['candidate'], 'k',          lw=0.8, label='candidate')
        ax.plot(t_s, h['mean'],      color='darkorange', lw=1.4,
                label='mean envelope')
        ax.set_ylabel(f"Iter {h['k']}", fontsize=9)
        ax.set_ylim(-1000, 1000)
        tag = '  [IMF accepted]' if h['sdk'] < stoplim else ''
        ax.set_title(f"sdk = {h['sdk']:.5f}{tag}", fontsize=9)

    axes[0].legend(loc='upper right', fontsize=8)
    axes[0].set_title(
        f'Sifting to extract IMF 1  (stoplim={stoplim:.0e})'
        f'  —  converged in {n_iter} step(s)',
        fontsize=10)
    axes[-1].set_xlabel('Time (s)')
    plt.tight_layout(); plt.show()
    return candidate, n_iter


imf1_verbose, n_steps = sifting_iterations(x, stoplim=1e-3)
print(f'IMF 1 extracted after {n_steps} sifting iteration(s).')


---
## 7. Sensitivity to the stopping criterion

The `stoplim` parameter is the main free parameter of EMD. How much does it matter?


In [ ]:
stoplims = [1e-2, 1e-3, 1e-4]
fig, axes = plt.subplots(len(stoplims), 1, figsize=(12, 7), sharex=True)

for ax, sl in zip(axes, stoplims):
    imfs_sl, _ = emd(x, n_imf=3, stoplim=sl)
    ax.plot(t, x,          color='0.8',       lw=0.6, label='original')
    ax.plot(t, imfs_sl[0], color='steelblue', lw=0.9,
            label=f'IMF 1  (stoplim={sl:.0e})')
    ax.set_ylabel('Voltage (µV)'); ax.set_ylim(-800, 800)
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Time (s)')
axes[0].set_title('Effect of stopping criterion on IMF 1')
plt.tight_layout(); plt.show()


**Reflection questions:**

1. How does the IMF shape change as `stoplim` decreases?
2. At what value does further tightening produce negligible change?
3. Why might a very small `stoplim` be undesirable in practice?


---
## 8. Reconstruction check

Summing all IMFs and the residual must recover $x(t)$ exactly (up to floating-point precision).


In [ ]:
N_IMF = 5
imfs, residual = emd(x, n_imf=N_IMF, stoplim=1e-3)
x_rec = sum(imfs) + residual

max_err = np.max(np.abs(x - x_rec))
print(f'Maximum reconstruction error: {max_err:.2e} µV')
assert max_err < 1e-6, 'Reconstruction error too large!'
print('Perfect reconstruction confirmed.')

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
axes[0].plot(t, x,     lw=0.9, label='original')
axes[0].plot(t, x_rec, lw=0.8, ls='--', label='reconstructed')
axes[0].legend(fontsize=9); axes[0].set_ylabel('Voltage (µV)')
axes[0].set_title('Original vs reconstructed (should overlap exactly)')

axes[1].plot(t, x - x_rec, color='crimson', lw=0.6)
axes[1].set_ylabel('Error (µV)'); axes[1].set_xlabel('Time (s)')
axes[1].set_title('Reconstruction error (should be ~0)')
plt.tight_layout(); plt.show()


---
## 9. Summary

| Concept | Key point |
|---------|----------|
| **Why EMD** | Non-stationary, nonlinear signals; no fixed basis needed |
| **IMF conditions** | Balanced extrema/zero-crossings; mean envelope ≈ 0 |
| **Sifting loop** | Iteratively subtract mean envelope until IMF condition met |
| **`stoplim`** | Free parameter; typical range $10^{-4}$–$10^{-2}$ |
| **Reconstruction** | $x(t) = \sum_i \text{IMF}_i(t) + r(t)$ exactly |
| **Hilbert–Huang** | Apply Hilbert transform to each IMF for $\omega_i(t)$ |

### Further reading

- Huang et al. (1998). *The empirical mode decomposition and the Hilbert spectrum for nonlinear and non-stationary time series analysis.*  
  Proc. R. Soc. London A, **454**, 903–995.
- Wu & Huang (2009). *Ensemble EMD: a noise-assisted data analysis method.*  
  Adv. Adapt. Data Anal., **1**(1), 1–41.
- Python packages: [`PyEMD`](https://pypi.org/project/EMD-signal/), [`emd`](https://emd.readthedocs.io/)
